# Catalog Flag Filter Diagnostics

用于快速检查 HSC/Scarlet catalog 中指定 flag 组合会过滤多少源，并统计被过滤源中其他 flags 的占比。默认读取 tract `9813`、patch `0,0`、band `HSC-I` 的原始 `meas` catalog。


In [6]:
from pathlib import Path
import warnings

import numpy as np
from astropy.table import Table

# ---- Catalog selection ----
TRACT = '9813'
PATCH = '0,0'
BAND = 'HSC-I'
CATALOG_ROOT = Path('/data1/czh23/Subaru')
CATALOG_PATH = None  # set a Path to override the tract/patch/band path
FITS_HDU = 1

# ---- Flag filter ----
# Removed rows are rows where ANY or ALL of these flags are true.
FILTER_FLAGS = [
    # 'base_PixelFlags_flag_bright_objectCenter',
    # 'base_PixelFlags_flag_saturatedCenter',
    # 'base_PixelFlags_flag_clippedCenter',
    # 'base_PixelFlags_flag_sensor_edgeCenter',
    # 'modelfit_CModel_flag_badCentroid',
    # 'modelfit_CModel_flag_region_maxBadPixelFraction',
    # 'base_SdssShape_flag_shift',
    'base_SdssShape_flag',
    'detect_isPrimary',
    'base_SdssCentroid_flag',
]
FILTER_MODE = 'any'  # 'any' or 'all'

# Optional non-flag base population masks.
REQUIRE_NCHILD0 = True
REQUIRE_FINITE_CMODEL_FLUX = True

# ---- Other-flag reporting among removed rows ----
REPORT_OTHER_FLAGS = True
EXCLUDE_FILTER_FLAGS_FROM_REPORT = True
TOP_N_OTHER_FLAGS = 40
MIN_OTHER_FLAG_FRACTION = 0.0

def resolved_catalog_path():
    if CATALOG_PATH is not None:
        return Path(CATALOG_PATH).expanduser()
    return CATALOG_ROOT / TRACT / BAND / PATCH / f'meas-{BAND}-{TRACT}-{PATCH}.fits'

catalog_path = resolved_catalog_path()
print('catalog:', catalog_path)


catalog: /data1/czh23/Subaru/9813/HSC-I/0,0/meas-HSC-I-9813-0,0.fits


In [7]:
def load_catalog(path: Path, hdu: int = 1):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        table = Table.read(path, hdu=hdu)
    truncated = any('truncated' in str(w.message).lower() for w in caught)
    if truncated:
        print('WARNING: FITS file may be truncated, but requested HDU was readable.')
    return table

def flag_name_to_bit(table: Table):
    return {str(value): int(key[len('TFLAG'):]) - 1 for key, value in table.meta.items() if key.startswith('TFLAG')}

def flag_values(table: Table, name: str, fmap=None, packed_flags=None):
    if name in table.colnames:
        return np.asarray(table[name], dtype=bool)
    if fmap is None:
        fmap = flag_name_to_bit(table)
    if packed_flags is None:
        packed_flags = np.asarray(table['flags'], dtype=bool)
    if name not in fmap:
        raise KeyError(f'Flag {name!r} not found. Example flags: {list(fmap)[:10]} ...')
    return np.asarray(packed_flags[:, fmap[name]], dtype=bool)

def optional_quality_mask(table: Table):
    mask = np.ones(len(table), dtype=bool)
    if REQUIRE_NCHILD0:
        child_col = 'deblend_nChild' if 'deblend_nChild' in table.colnames else 'nChild' if 'nChild' in table.colnames else None
        if child_col is None:
            raise KeyError('REQUIRE_NCHILD0 needs deblend_nChild or nChild')
        mask &= np.asarray(table[child_col], dtype=np.int64) == 0
    if REQUIRE_FINITE_CMODEL_FLUX:
        if 'modelfit_CModel_instFlux' not in table.colnames:
            raise KeyError('REQUIRE_FINITE_CMODEL_FLUX needs modelfit_CModel_instFlux')
        flux = np.asarray(table['modelfit_CModel_instFlux'], dtype=float)
        mask &= np.isfinite(flux) & (flux > 0)
    return mask

def build_removed_mask(table: Table, filter_flags, mode='any'):
    fmap = flag_name_to_bit(table)
    packed_flags = np.asarray(table['flags'], dtype=bool) if 'flags' in table.colnames else None
    if not filter_flags:
        flag_mask = np.zeros(len(table), dtype=bool)
    else:
        values = [flag_values(table, name, fmap, packed_flags) for name in filter_flags]
        stacked = np.stack(values, axis=0)
        if mode == 'any':
            flag_mask = stacked.any(axis=0)
        elif mode == 'all':
            flag_mask = stacked.all(axis=0)
        else:
            raise ValueError("FILTER_MODE must be 'any' or 'all'")
    base = optional_quality_mask(table)
    # Count filtering inside the optional base population.
    removed = base & flag_mask
    kept = base & ~flag_mask
    return removed, kept, base, fmap, packed_flags

table = load_catalog(catalog_path, FITS_HDU)
removed, kept, base, fmap, packed_flags = build_removed_mask(table, FILTER_FLAGS, FILTER_MODE)
print('rows total:', len(table))
print('base population:', int(base.sum()))
print('removed:', int(removed.sum()), f'({removed.sum() / max(base.sum(), 1):.3%} of base)')
print('kept:', int(kept.sum()), f'({kept.sum() / max(base.sum(), 1):.3%} of base)')


rows total: 27408
base population: 22012
removed: 7390 (33.573% of base)
kept: 14622 (66.427% of base)


## A/B Filter Region Export

可选导出一个 DS9 region 文件，用两种颜色对比两套过滤规则：

- green：通过 Filter A 且通过 Filter B 的源。
- red：通过 Filter A 但被 Filter B 删除的源。

这适合比较“基础 clean 规则 A”与“新增严格规则 B”的差异。


In [8]:
# ---- Optional A/B region export ----
EXPORT_AB_REG = True
REGION_OUTPUT_DIR = Path('output/catalog_flag_filter_diagnostics')
REGION_ELLIPSE_SIGMA = 3.0
REGION_MIN_AXIS = 1.5

# A is the baseline filter, B is the stricter/additional filter.
# A+B kept means rows not removed by A and not removed by B.
# A kept / B removed means rows that pass A but fail B.
FILTER_A_FLAGS = [
    # 'base_PixelFlags_flag_saturatedCenter',
    
    'modelfit_CModel_flag_badCentroid',
    'base_InputCount_flag',
    
    # B-class flags: excluded from labels
    
]
FILTER_A_MODE = 'any'
FILTER_B_FLAGS = [
    'base_PixelFlags_flag_clippedCenter',
    'base_SdssShape_flag_shift',
    'base_SdssCentroid_flag_badError',
    'ext_photometryKron_KronFlux_flag_edge',
    'base_NaiveCentroid_flag_edge',
    'base_PsfFlux_flag_edge',
    'base_PixelFlags_flag_edge',
    'base_SdssCentroid_flag_edge',
    # Experimenting with additional flags to add to B-class filter. Exclude from labels if included.
    'modelfit_CModel_flag_region_maxBadPixelFraction',
    'ext_photometryKron_KronFlux_flag_small_radius',
    # 'modelfit_CModel_flags_smallShape',
    'base_NaiveCentroid_flag_noCounts',
    
    'ext_photometryKron_KronFlux_flag_bad_shape',
    'deblend_rampedTemplate',
    'deblend_patchedTemplate',
    'deblend_deblendedAsPsf',
    'deblend_tooManyPeaks',
    'deblend_skipped',
    'deblend_rampedTemplate',
    'deblend_patchedTemplate',
    'deblend_masked',

    # 'modelfit_CModel_flags_region_usedFootprintArea',
    # 'base_PixelFlags_flag_sensor_edgeCenter',
    # 'base_PixelFlags_flag_bright_object'
    # 'base_PixelFlags_flag_bright_objectCenter',
    # 'modelfit_CModel_flag_badCentroid',
    # 'modelfit_CModel_flag_region_maxBadPixelFraction',
    # 'base_SdssShape_flag_shift',
]
FILTER_B_MODE = 'any'

# Optional numeric B cuts. Set to None to disable.
FILTER_B_ELLIPSE_AREA_MAX = None  # example: 900.0 removes rows with ellipse_area_3sigma >= 900
FILTER_B_FOOTPRINT_AREA_MAX = None  # example: 900.0 removes rows with base_FootprintArea_value > 900
# Debug cut: remove rows whose Kron radius is suspiciously small relative to the SDSS semi-major axis.
FILTER_B_KRON_RADIUS_LT_SDSS_MAJOR_RATIO = 0.75

REG_COLOR_AB_KEPT = 'green'
REG_COLOR_A_KEPT_B_REMOVED = 'red'

# Very large ellipses make DS9 views unreadable. If pi*a*b exceeds this
# display-area threshold, write only a center point instead of an ellipse.
REG_LARGE_ELLIPSE_AREA_AS_POINT = 40000.0
REG_COLOR_AB_KEPT_LARGE = 'cyan'
REG_COLOR_A_KEPT_B_REMOVED_LARGE = 'magenta'
REG_POINT_SIZE = 8

# Optional Kron-shape region export using the configured A/B filters above.
# The output mirrors the SDSS A/B region file: A+B kept and A-kept/B-removed
# are written into one DS9 file with different colors.
EXPORT_KRON_SHAPE_REG = True


In [9]:
# Per-filter-flag counts inside the base population.
rows = []
for name in FILTER_FLAGS:
    vals = flag_values(table, name, fmap, packed_flags)
    count = int(np.count_nonzero(base & vals))
    rows.append((name, count, count / max(int(base.sum()), 1)))

print('Filter flags inside base population:')
for name, count, frac in rows:
    print(f'{count:7d}  {frac:8.3%}  {name}')


Filter flags inside base population:
  10689   39.000%  base_PixelFlags_flag_bright_objectCenter
    812    2.963%  base_PixelFlags_flag_saturatedCenter
    836    3.050%  base_PixelFlags_flag_clippedCenter
  10979   40.058%  base_PixelFlags_flag_sensor_edgeCenter
      0    0.000%  modelfit_CModel_flag_badCentroid
   1685    6.148%  modelfit_CModel_flag_region_maxBadPixelFraction
    661    2.412%  base_SdssShape_flag_shift


In [10]:
def all_flag_names(table: Table):
    return [name for _bit, name in sorted((bit, name) for name, bit in flag_name_to_bit(table).items())]

def other_flag_fraction_table(table: Table, removed_mask, *, exclude_filter_flags=True):
    names = all_flag_names(table)
    if exclude_filter_flags:
        names = [name for name in names if name not in set(FILTER_FLAGS)]
    denom = int(np.count_nonzero(removed_mask))
    out = []
    if denom == 0:
        return out
    for name in names:
        vals = flag_values(table, name, fmap, packed_flags)
        count = int(np.count_nonzero(removed_mask & vals))
        frac = count / denom
        if count > 0 and frac >= MIN_OTHER_FLAG_FRACTION:
            doc = table.meta.get(f'TFDOC{fmap[name] + 1}', '')
            out.append((name, count, frac, str(doc)))
    out.sort(key=lambda row: (-row[1], row[0]))
    return out

if REPORT_OTHER_FLAGS:
    other = other_flag_fraction_table(table, removed, exclude_filter_flags=EXCLUDE_FILTER_FLAGS_FROM_REPORT)
    print(f'Other flags among removed rows: top {TOP_N_OTHER_FLAGS}')
    print(f'removed rows denominator: {int(removed.sum())}')
    for name, count, frac, doc in other[:TOP_N_OTHER_FLAGS]:
        print(f'{count:7d}  {frac:8.3%}  {name}  # {doc}')


Other flags among removed rows: top 40
removed rows denominator: 18241
  18241  100.000%  ext_convolved_ConvolvedFlux_0_deconv  # deconvolution required for seeing 3.500000; no measurement made
  17618   96.585%  merge_footprint_i  # Detection footprint overlapped with a detection from filter i
  17547   96.195%  base_PixelFlags_flag_inexact_psf  # Source footprint includes INEXACT_PSF pixels
  17345   95.088%  detect_isPatchInner  # true if source is in the inner region of a coadd patch
  16623   91.130%  merge_footprint_z  # Detection footprint overlapped with a detection from filter z
  16539   90.669%  merge_footprint_g  # Detection footprint overlapped with a detection from filter g
  16482   90.357%  modelfit_CModel_flags_region_usedInitialEllipseMax  # the pixel region for the final fit was set to the upper bound defined by the initial fit
  16464   90.258%  modelfit_CModel_initial_flag_trSmall  # the optimizer converged because the trust radius became too small; this is a less-

In [11]:
# Optional: compare other-flag fractions between removed and kept populations.
COMPARE_REMOVED_KEPT = True
TOP_N_COMPARE = 40

if COMPARE_REMOVED_KEPT:
    denom_removed = int(removed.sum())
    denom_kept = int(kept.sum())
    comp = []
    for name in all_flag_names(table):
        if EXCLUDE_FILTER_FLAGS_FROM_REPORT and name in set(FILTER_FLAGS):
            continue
        vals = flag_values(table, name, fmap, packed_flags)
        r_count = int(np.count_nonzero(removed & vals))
        k_count = int(np.count_nonzero(kept & vals))
        r_frac = r_count / max(denom_removed, 1)
        k_frac = k_count / max(denom_kept, 1)
        if r_count or k_count:
            comp.append((name, r_count, r_frac, k_count, k_frac, r_frac - k_frac))
    comp.sort(key=lambda row: (-abs(row[-1]), row[0]))
    print('Flag enrichment: removed vs kept')
    print(' removed_count removed_frac kept_count kept_frac delta  flag')
    for name, rc, rf, kc, kf, delta in comp[:TOP_N_COMPARE]:
        print(f'{rc:7d} {rf:8.3%} {kc:7d} {kf:8.3%} {delta:8.3%}  {name}')


Flag enrichment: removed vs kept
 removed_count removed_frac kept_count kept_frac delta  flag
  11103  60.868%     551   6.011%  54.858%  base_PixelFlags_flag_bright_object
  14774  80.993%    3250  35.453%  45.540%  base_PixelFlags_flag_inexact_psfCenter
  16001  87.720%    6460  70.470%  17.250%  base_PixelFlags_flag_sensor_edge
   2745  15.049%     416   4.538%  10.511%  base_ClassificationExtendedness_flag
  16250  89.085%    9099  99.258% -10.173%  modelfit_CModel_dev_flag_trSmall
  16295  89.332%    9110  99.378% -10.046%  modelfit_CModel_exp_flag_trSmall
   1911  10.476%      50   0.545%   9.931%  modelfit_CModel_dev_flag
   1912  10.482%      51   0.556%   9.926%  modelfit_CModel_flag
   1871  10.257%      43   0.469%   9.788%  modelfit_CModel_exp_flag
  16464  90.258%    9123  99.520%  -9.262%  modelfit_CModel_initial_flag_trSmall
  16482  90.357%    9120  99.487%  -9.130%  modelfit_CModel_flags_region_usedInitialEllipseMax
   1616   8.859%       8   0.087%   8.772%  modelfit_

In [12]:
def _finite_col(table: Table, names):
    for name in names:
        if name in table.colnames:
            return np.asarray(table[name], dtype=float)
    return np.full(len(table), np.nan, dtype=float)

def _sdss_ellipse_params(table: Table, sigma: float = 3.0):
    xx = _finite_col(table, ('base_SdssShape_xx', 'ext_shapeHSM_HsmSourceMoments_xx'))
    yy = _finite_col(table, ('base_SdssShape_yy', 'ext_shapeHSM_HsmSourceMoments_yy'))
    xy = _finite_col(table, ('base_SdssShape_xy', 'ext_shapeHSM_HsmSourceMoments_xy'))
    major = np.full(len(table), np.nan, dtype=float)
    minor = np.full(len(table), np.nan, dtype=float)
    angle = np.full(len(table), np.nan, dtype=float)
    area = np.full(len(table), np.nan, dtype=float)
    for i, (vxx, vyy, vxy) in enumerate(zip(xx, yy, xy)):
        if not (np.isfinite(vxx) and np.isfinite(vyy) and np.isfinite(vxy)):
            continue
        cov = np.array([[vxx, vxy], [vxy, vyy]], dtype=float)
        vals, vecs = np.linalg.eigh(cov)
        if not np.all(np.isfinite(vals)) or vals[0] <= 0 or vals[1] <= 0:
            continue
        order = np.argsort(vals)[::-1]
        vals = vals[order]
        vec = vecs[:, order[0]]
        major[i] = np.sqrt(vals[0])
        minor[i] = np.sqrt(vals[1])
        angle[i] = np.degrees(np.arctan2(vec[1], vec[0]))
        area[i] = np.pi * (sigma * major[i]) * (sigma * minor[i])
    return major, minor, angle, area

def _kron_ellipse_params(table: Table, sigma: float = 3.0):
    sdss_major, sdss_minor, sdss_angle, _sdss_area = _sdss_ellipse_params(table, sigma=sigma)
    kron_radius = _finite_col(
        table,
        (
            'ext_photometryKron_KronFlux_radius',
            'ext_photometryKron_KronFlux_radius_for_radius',
        ),
    )
    axis_ratio = np.clip(sdss_minor / np.maximum(sdss_major, 1e-6), 0.15, 1.0)
    valid = np.isfinite(kron_radius) & (kron_radius > 0) & np.isfinite(axis_ratio) & np.isfinite(sdss_angle)
    major = np.full(len(table), np.nan, dtype=float)
    minor = np.full(len(table), np.nan, dtype=float)
    angle = np.full(len(table), np.nan, dtype=float)
    area = np.full(len(table), np.nan, dtype=float)
    major[valid] = kron_radius[valid]
    minor[valid] = kron_radius[valid] * axis_ratio[valid]
    angle[valid] = sdss_angle[valid]
    area[valid] = np.pi * (sigma * major[valid]) * (sigma * minor[valid])
    return major, minor, angle, area

def _filter_removed_mask_for_flags(table: Table, flags_list, mode, fmap, packed_flags):
    if not flags_list:
        return np.zeros(len(table), dtype=bool)
    vals = [flag_values(table, name, fmap, packed_flags) for name in flags_list]
    stacked = np.stack(vals, axis=0)
    if mode == 'any':
        return stacked.any(axis=0)
    if mode == 'all':
        return stacked.all(axis=0)
    raise ValueError("mode must be 'any' or 'all'")

def _write_ab_region(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        handle.write('# Region file format: DS9 version 4.1\n')
        handle.write('global color=green dashlist=8 3 width=2 font="helvetica 14 bold roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1\n')
        handle.write('image\n')
        for row in rows:
            if row.get('kind') == 'point':
                handle.write(
                    f"point({row['x'] + 1.0:.3f},{row['y'] + 1.0:.3f}) # point=circle {int(row.get('size', REG_POINT_SIZE))} color={row['color']}\n"
                )
            else:
                handle.write(
                    f"ellipse({row['x'] + 1.0:.3f},{row['y'] + 1.0:.3f},{row['a']:.3f},{row['b']:.3f},{row['angle']:.2f}) # color={row['color']}\n"
                )

def _compute_ab_filter_masks(table: Table):
    fmap_local = flag_name_to_bit(table)
    packed_local = np.asarray(table['flags'], dtype=bool) if 'flags' in table.colnames else None
    base_mask = optional_quality_mask(table)

    removed_a = _filter_removed_mask_for_flags(table, FILTER_A_FLAGS, FILTER_A_MODE, fmap_local, packed_local)
    removed_b = _filter_removed_mask_for_flags(table, FILTER_B_FLAGS, FILTER_B_MODE, fmap_local, packed_local)

    sdss_major, sdss_minor, sdss_angle, sdss_ellipse_area = _sdss_ellipse_params(table, sigma=REGION_ELLIPSE_SIGMA)
    if FILTER_B_ELLIPSE_AREA_MAX is not None:
        removed_b |= np.isfinite(sdss_ellipse_area) & (sdss_ellipse_area >= float(FILTER_B_ELLIPSE_AREA_MAX))
    if FILTER_B_FOOTPRINT_AREA_MAX is not None:
        footprint = _finite_col(table, ('base_FootprintArea_value',))
        removed_b |= np.isfinite(footprint) & (footprint > float(FILTER_B_FOOTPRINT_AREA_MAX))

    kron_small_vs_sdss = np.zeros(len(table), dtype=bool)
    if FILTER_B_KRON_RADIUS_LT_SDSS_MAJOR_RATIO is not None:
        kron_radius = _finite_col(
            table,
            (
                'ext_photometryKron_KronFlux_radius',
                'ext_photometryKron_KronFlux_radius_for_radius',
            ),
        )
        ratio = float(FILTER_B_KRON_RADIUS_LT_SDSS_MAJOR_RATIO)
        kron_small_vs_sdss = (
            np.isfinite(kron_radius)
            & np.isfinite(sdss_major)
            & (sdss_major > 0)
            & (kron_radius < ratio * sdss_major)
        )
        removed_b |= kron_small_vs_sdss

    return {
        'base_mask': base_mask,
        'removed_a': removed_a,
        'removed_b': removed_b,
        'kron_small_vs_sdss': kron_small_vs_sdss,
        'sdss_major': sdss_major,
        'sdss_minor': sdss_minor,
        'sdss_angle': sdss_angle,
        'sdss_ellipse_area': sdss_ellipse_area,
    }


def export_ab_filter_region():
    masks = _compute_ab_filter_masks(table)
    base_mask = masks['base_mask']
    removed_a = masks['removed_a']
    removed_b = masks['removed_b']
    kron_small_vs_sdss = masks['kron_small_vs_sdss']

    x = _finite_col(table, ('base_SdssCentroid_x', 'base_SdssShape_x', 'base_NaiveCentroid_x', 'deblend_psfCenter_x'))
    y = _finite_col(table, ('base_SdssCentroid_y', 'base_SdssShape_y', 'base_NaiveCentroid_y', 'deblend_psfCenter_y'))
    major, minor, angle, ellipse_area = _sdss_ellipse_params(table, sigma=REGION_ELLIPSE_SIGMA)

    valid_shape = (
        base_mask
        & np.isfinite(x) & np.isfinite(y)
        & np.isfinite(major) & np.isfinite(minor) & np.isfinite(angle)
        & (major > 0) & (minor > 0)
    )
    ab_kept = valid_shape & ~removed_a & ~removed_b
    a_kept_b_removed = valid_shape & ~removed_a & removed_b

    rows = []
    large_counts = {REG_COLOR_AB_KEPT_LARGE: 0, REG_COLOR_A_KEPT_B_REMOVED_LARGE: 0}
    specs = [
        (ab_kept, REG_COLOR_AB_KEPT, REG_COLOR_AB_KEPT_LARGE),
        (a_kept_b_removed, REG_COLOR_A_KEPT_B_REMOVED, REG_COLOR_A_KEPT_B_REMOVED_LARGE),
    ]
    for mask, color, large_color in specs:
        for i in np.flatnonzero(mask):
            a = max(abs(float(major[i])) * REGION_ELLIPSE_SIGMA, REGION_MIN_AXIS)
            b = max(abs(float(minor[i])) * REGION_ELLIPSE_SIGMA, REGION_MIN_AXIS)
            display_area = float(np.pi * a * b)
            if np.isfinite(display_area) and display_area > float(REG_LARGE_ELLIPSE_AREA_AS_POINT):
                rows.append({
                    'kind': 'point',
                    'x': float(x[i]),
                    'y': float(y[i]),
                    'color': large_color,
                    'size': REG_POINT_SIZE,
                })
                large_counts[large_color] = large_counts.get(large_color, 0) + 1
            else:
                rows.append({
                    'kind': 'ellipse',
                    'x': float(x[i]),
                    'y': float(y[i]),
                    'a': a,
                    'b': b,
                    'angle': float(angle[i]),
                    'color': color,
                })

    stem = Path(catalog_path).stem
    out_path = REGION_OUTPUT_DIR / f'{stem}_filterA_kept_filterB_compare.reg'
    _write_ab_region(out_path, rows)
    print('saved:', out_path)
    print('A+B kept:', int(np.count_nonzero(ab_kept)), f'ellipse_color={REG_COLOR_AB_KEPT}', f'large_point_color={REG_COLOR_AB_KEPT_LARGE}')
    print('A kept, B removed:', int(np.count_nonzero(a_kept_b_removed)), f'ellipse_color={REG_COLOR_A_KEPT_B_REMOVED}', f'large_point_color={REG_COLOR_A_KEPT_B_REMOVED_LARGE}')
    if FILTER_B_KRON_RADIUS_LT_SDSS_MAJOR_RATIO is not None:
        print(
            'B kron_radius < ratio * sdss_major:',
            int(np.count_nonzero(kron_small_vs_sdss & valid_shape & ~removed_a)),
            f'ratio={float(FILTER_B_KRON_RADIUS_LT_SDSS_MAJOR_RATIO):g}',
        )
    print('large ellipses written as points:', large_counts)
    return out_path

def export_kron_shape_region():
    masks = _compute_ab_filter_masks(table)
    x = _finite_col(table, ('base_SdssCentroid_x', 'base_SdssShape_x', 'base_NaiveCentroid_x', 'deblend_psfCenter_x'))
    y = _finite_col(table, ('base_SdssCentroid_y', 'base_SdssShape_y', 'base_NaiveCentroid_y', 'deblend_psfCenter_y'))
    major, minor, angle, kron_area = _kron_ellipse_params(table, sigma=REGION_ELLIPSE_SIGMA)
    valid_shape = (
        masks['base_mask']
        & np.isfinite(x) & np.isfinite(y)
        & np.isfinite(major) & np.isfinite(minor) & np.isfinite(angle)
        & (major > 0) & (minor > 0)
    )
    ab_kept = valid_shape & ~masks['removed_a'] & ~masks['removed_b']
    a_kept_b_removed = valid_shape & ~masks['removed_a'] & masks['removed_b']

    rows = []
    large_counts = {REG_COLOR_AB_KEPT_LARGE: 0, REG_COLOR_A_KEPT_B_REMOVED_LARGE: 0}
    specs = [
        (ab_kept, REG_COLOR_AB_KEPT, REG_COLOR_AB_KEPT_LARGE),
        (a_kept_b_removed, REG_COLOR_A_KEPT_B_REMOVED, REG_COLOR_A_KEPT_B_REMOVED_LARGE),
    ]
    for mask, color, large_color in specs:
        for i in np.flatnonzero(mask):
            a = max(abs(float(major[i])) * REGION_ELLIPSE_SIGMA, REGION_MIN_AXIS)
            b = max(abs(float(minor[i])) * REGION_ELLIPSE_SIGMA, REGION_MIN_AXIS)
            display_area = float(np.pi * a * b)
            if np.isfinite(display_area) and display_area > float(REG_LARGE_ELLIPSE_AREA_AS_POINT):
                rows.append({
                    'kind': 'point',
                    'x': float(x[i]),
                    'y': float(y[i]),
                    'color': large_color,
                    'size': REG_POINT_SIZE,
                })
                large_counts[large_color] = large_counts.get(large_color, 0) + 1
            else:
                rows.append({
                    'kind': 'ellipse',
                    'x': float(x[i]),
                    'y': float(y[i]),
                    'a': a,
                    'b': b,
                    'angle': float(angle[i]),
                    'color': color,
                })

    stem = Path(catalog_path).stem
    out_path = REGION_OUTPUT_DIR / f'{stem}_kron_shape_filterA_kept_filterB_compare.reg'
    _write_ab_region(out_path, rows)
    print('saved kron shape:', out_path)
    print('Kron A+B kept:', int(np.count_nonzero(ab_kept)), f'ellipse_color={REG_COLOR_AB_KEPT}', f'large_point_color={REG_COLOR_AB_KEPT_LARGE}')
    print('Kron A kept, B removed:', int(np.count_nonzero(a_kept_b_removed)), f'ellipse_color={REG_COLOR_A_KEPT_B_REMOVED}', f'large_point_color={REG_COLOR_A_KEPT_B_REMOVED_LARGE}')
    print('Kron large ellipses written as points:', large_counts)
    return out_path


if EXPORT_AB_REG:
    ab_region_path = export_ab_filter_region()
else:
    print('Set EXPORT_AB_REG = True to write the A/B comparison DS9 region file.')

if EXPORT_KRON_SHAPE_REG:
    kron_shape_region_path = export_kron_shape_region()
else:
    print('Set EXPORT_KRON_SHAPE_REG = True to write the filtered Kron-shape DS9 region file.')


saved: output/catalog_flag_filter_diagnostics/meas-HSC-I-9813-0,0_filterA_kept_filterB_compare.reg
A+B kept: 7776 ellipse_color=green large_point_color=cyan
A kept, B removed: 18919 ellipse_color=red large_point_color=magenta
B kron_radius < ratio * sdss_major: 1100 ratio=0.75
large ellipses written as points: {'cyan': 0, 'magenta': 418}
saved kron shape: output/catalog_flag_filter_diagnostics/meas-HSC-I-9813-0,0_kron_shape_filterA_kept_filterB_compare.reg
Kron A+B kept: 7776 ellipse_color=green large_point_color=cyan
Kron A kept, B removed: 18284 ellipse_color=red large_point_color=magenta
Kron large ellipses written as points: {'cyan': 1, 'magenta': 65}


## 常用配置例子

严格 clean 过滤可以设置：

```python
FILTER_FLAGS = [
    'base_PixelFlags_flag_bright_objectCenter',
    'base_PixelFlags_flag_saturatedCenter',
    'base_PixelFlags_flag_clippedCenter',
    'base_PixelFlags_flag_sensor_edgeCenter',
    'modelfit_CModel_flag_badCentroid',
    'modelfit_CModel_flag_region_maxBadPixelFraction',
    'base_SdssShape_flag_shift',
]
FILTER_MODE = 'any'
REQUIRE_NCHILD0 = True
REQUIRE_FINITE_CMODEL_FLUX = True
```

如果只想看单个 flag 的影响，例如 bright-object center：

```python
FILTER_FLAGS = ['base_PixelFlags_flag_bright_objectCenter']
FILTER_MODE = 'any'
```
